# Syncing CSV data to the SQLite DB
<hr>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import requests
import seaborn as sns
from sklearn.linear_model import LinearRegression
import Data_Functions as dh
import DataHandler as dbh

## Setting up Database Handler
<hr>

In [2]:
db = dbh.Database_Handler()

### Data handler stores game results as: <br>
Game Version, Date, Time Survived, Score
<br><br>
### Data handler stores game end data as: <br>
Date, Collision Object, Time Object Spawned, Enemies Alive
<br><br>
### Data handler stores powerups spawned as: <br>
Date, Powerup (name), Time Spawned, Time Activated, Time Effect Ended, Times Player Hit


In [3]:
gr_df = dh.read_results()

ged_df = dh.read_game_end_data()

gr_powerups = dh.read_powerup_data()

### Changes required for Game Results and Game End Data to become GameRun: <br>
<ul>
    <li>add score_per_second column</li>
    <li>merge gr_df and ged_df</li>
</ul>

### Changes requred for Powerup data to become PowerupsSpawned: <br>
<ul>
    <li>Add PowerupID for each Powerup (name)</li>
    <li>Remove Powerup (name)</li>
</ul>

## Preparing Game Results and Game End Data 
<hr>

In [4]:
gr_df["Score Per Second"] = gr_df["Score"] / gr_df["Time Survived"]

In [5]:
game_run_df = pd.merge(gr_df, ged_df, on="Date", how="left")
game_run_df = game_run_df.replace({pd.NA: None})

In [6]:
game_run_df

,Game Version,Time Survived,Score,Survival Group,Score Group,Score Per Second,Collision Object,Time Object Spawned,Enemies Alive
Date,,,,,,,,,
2026-08-11 22:02:53.976116,2026-08-10 23:55:20.054130,10.757392,260,10 < 30,100 < 499,24.169427,None,NaN,NaN
2026-08-11 22:03:49.405594,2026-08-10 23:55:20.054130,24.408522,933,10 < 30,500 < 999,38.224355,None,NaN,NaN
2026-08-11 22:07:52.788656,2026-08-10 23:55:20.054130,1.700332,31,0 < 5,0 < 99,18.231729,None,NaN,NaN
2026-08-11 22:08:16.921228,2026-08-10 23:55:20.054130,12.781978,340,10 < 30,100 < 499,26.599951,None,NaN,NaN
2026-08-11 22:09:49.791919,2026-08-10 23:55:20.054130,0.995767,17,0 < 5,0 < 99,17.072269,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-09-15 18:41:42.283207,2026-09-15 16:50:42.768660,17.247342,528,10 < 30,500 < 999,30.613412,Cube,1.587924,21.0
2026-09-15 18:42:00.192944,2026-09-15 16:50:42.768660,15.583310,449,10 < 30,100 < 499,28.812877,Cube,13.362484,21.0
2026-09-16 19:23:16.080657,2026-09-13 20:34:46.245973,2.950675,53,0 < 5,0 < 99,17.96199,Cube,1.924126,20.0


## Preparing Powerups Data for PowerupsSpawned
<hr>

In [7]:
gr_powerups["PowerupID"] = gr_powerups["Powerup"].apply(db.get_powerup)

In [8]:
gr_powerups

,Powerup,Time Spawned,Time Activated,Time Effect Ended,Times Player Hit,PowerupID
Date,,,,,,
2026-08-20 19:53:15.689284,Eat_Enemy_Powerup,22.549473,25.925230,30.938932,NaN,2
2026-08-20 19:53:15.689284,Eat_Enemy_Powerup,45.070239,45.870412,50.875540,NaN,2
2026-08-20 19:53:15.689284,Score_Increase_Powerup,15.043289,15.925274,20.856675,NaN,1
2026-08-20 19:53:15.689284,Score_Increase_Powerup,30.052839,30.883667,35.842999,NaN,1
2026-08-20 19:53:15.689284,Temp_Change_Colour_Powerup,7.528527,NaN,NaN,NaN,3
...,...,...,...,...,...,...
2026-09-16 19:24:26.477879,Temp_Change_Colour_Powerup,7.515464,11.126915,NaN,7.0,3
2026-09-16 19:25:17.348551,Eat_Enemy_Powerup,15.019709,15.893578,20.899181,5.0,2
2026-09-16 19:25:17.348551,Eat_Enemy_Powerup,22.524488,24.291256,29.299025,3.0,2


## Pre insert db sizes
<hr>
GameRun

In [9]:
before_gr_count = db.get_GameRun_count()
before_gr_count

66

PowerupsSpawned

In [10]:
before_ps_count = db.get_PowerupsSpawned_count()
before_ps_count

34

## Adding data frames to db
<hr>

In [11]:
next(game_run_df.itertuples())

Pandas(Index=Timestamp('2026-08-11 22:02:53.976116'), _1=Timestamp('2026-08-10 23:55:20.054130'), _2=10.757391929626465, Score=np.int64(260), _4='10 < 30', _5='100 < 499', _6=24.169427097282316, _7=None, _8=nan, _9=nan)

In [12]:
cur = db.db.cursor()
text = "INSERT OR IGNORE INTO GameRun (date, game_version_time, time_survived, score, score_per_second, collision_object, time_object_spawned, enemies_alive) VALUES "
for row in game_run_df.itertuples():
    text += f"('{row.Index}', '{row._1}', '{row._2}', '{row.Score}', " 
    text += f"{'NULL' if row._6 is None else row._6}, '{'NULL' if row._7 is None else row._7}', '{'NULL' if pd.isna(row._8) else row._8}', '{'NULL' if pd.isna(row._9) else row._9}'), "
text = text[:-2]
text = text.replace("'NULL'", "NULL")
#print(text)
cur.execute(text)
db.db.commit()

In [13]:
next(gr_powerups.itertuples())

Pandas(Index=Timestamp('2026-08-20 19:53:15.689284'), Powerup='Eat_Enemy_Powerup', _2=22.54947328567505, _3=25.925230264663696, _4=30.938931703567505, _5=nan, PowerupID=2)

In [14]:
text = "INSERT OR IGNORE INTO PowerupsSpawned (date, powerupID, time_spawned, time_activated, time_ended) VALUES "
for row in gr_powerups.itertuples():
    text += f"('{row.Index}', '{row.PowerupID}', '{row._2}', '{'NULL' if pd.isna(row._3) else row._3}', '{'NULL' if pd.isna(row._4) else row._4}'), "
text = text[:-2]
text = text.replace("'NULL'", "NULL")
#print(text)
cur.execute(text)
db.db.commit()

INSERT OR IGNORE INTO PowerupsSpawned (date, powerupID, time_spawned, time_activated, time_ended) VALUES ('2026-08-20 19:53:15.689284', '2', '22.54947328567505', '25.925230264663696', '30.938931703567505'), ('2026-08-20 19:53:15.689284', '2', '45.07023859024048', '45.87041187286377', '50.87553954124451'), ('2026-08-20 19:53:15.689284', '1', '15.043289184570312', '15.925274133682253', '20.856674671173096'), ('2026-08-20 19:53:15.689284', '1', '30.052839279174805', '30.883666515350345', '35.84299850463867'), ('2026-08-20 19:53:15.689284', '3', '7.528526544570923', NULL, NULL), ('2026-08-20 19:53:15.689284', '3', '37.566134452819824', '42.25059151649475', '47.2196044921875'), ('2026-08-20 19:53:15.689284', '3', '52.58163619041443', '54.18901181221008', NULL), ('2026-08-20 19:57:41.418609', '1', '15.034730911254885', '18.964815378189087', '23.95594358444214'), ('2026-08-20 19:57:41.418609', '3', '7.519971609115601', '9.221335649490356', '14.218408584594728'), ('2026-08-20 19:57:41.418609',

## Post insert db sizes
<hr>
GameRun

In [15]:
after_gr_count = db.get_GameRun_count()
after_gr_count

235

In [16]:
difference = after_gr_count - before_gr_count
difference

169

PowerupsSpawned

In [17]:
after_ps_count = db.get_PowerupsSpawned_count()
after_ps_count

337

In [18]:
difference = after_ps_count - before_ps_count
difference

303